# Module 07 — Biological Interpretation

This notebook visualizes pre-computed pathway enrichment, transcription factor
activity, and pain gene analyses from `scripts/07_interpretation.py`. These
results build on the DE genes from Module 06 to provide biological context.

**Key findings:**
- **1,577 significant ORA terms** across cell types (GO:BP, GO:MF, GO:CC, KEGG, Reactome)
- **1,576 significant GSEA terms** using IVD-specific custom gene sets
- **290 transcription factors** tested; multiple significantly altered in degeneration
- **10 significant pain-related genes** differentially expressed in degeneration

**Data sources:**
- `results/interpretation/pathway_enrichment/` — ORA dot plots and GSEA results
- `results/interpretation/tf_activity/` — TF activity heatmap and results
- `results/interpretation/pain_genes.tsv` — Pain gene DE results
- `results/interpretation/pain_genes_heatmap.png` — Pain gene visualization

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})

# ── Paths ──────────────────────────────────────────────────────────────────
BASE = Path('..').resolve()
RESULTS = BASE / 'results' / 'interpretation'
PATHWAY_DIR = RESULTS / 'pathway_enrichment'
TF_DIR = RESULTS / 'tf_activity'

print(f'Results directory: {RESULTS}')
print(f'Pathway plots:    {len(list(PATHWAY_DIR.glob("*.png")))} files')
print(f'TF files:         {len(list(TF_DIR.glob("*")))} files')

## Over-Representation Analysis (ORA)

ORA tests whether DE gene lists are enriched for known biological pathways
(GO, KEGG, Reactome). Separate analyses for up- and down-regulated genes in
each cell type.

### Summary Statistics

In [ ]:
# Load all enrichment results
enrich_path = PATHWAY_DIR / 'all_enrichment_results.tsv'
if enrich_path.exists():
    enrich = pd.read_csv(enrich_path, sep='\t')
    print(f'Total enrichment results: {len(enrich)}')
    
    # Show summary by cell type and direction
    if 'cell_type' in enrich.columns and 'direction' in enrich.columns:
        summary = enrich.groupby(['cell_type', 'direction']).size().reset_index(name='n_terms')
        display(summary.style.set_caption('ORA Enrichment Terms by Cell Type and Direction'))
    elif 'source' in enrich.columns:
        summary = enrich.groupby('source').size().reset_index(name='n_terms')
        display(summary)
    
    # Show top terms
    if 'p_value' in enrich.columns:
        top = enrich.nsmallest(20, 'p_value')
        cols = [c for c in ['cell_type', 'direction', 'name', 'source', 'p_value'] if c in top.columns]
        display(top[cols].style.set_caption('Top 20 Most Significant ORA Terms'))
else:
    print('Enrichment results file not found')

### ORA Dot Plots — Upregulated Genes

In [ ]:
# Display ORA plots for upregulated genes
up_plots = sorted(PATHWAY_DIR.glob('enrichment_*_up.png'))
print(f'Upregulated enrichment plots: {len(up_plots)}')

for fpath in up_plots:
    label = fpath.stem.replace('enrichment_', '').replace('_up', ' (UP)').replace('_', ' ')
    display(Markdown(f'**{label}**'))
    display(Image(filename=str(fpath), width=800))

### ORA Dot Plots — Downregulated Genes

In [ ]:
# Display ORA plots for downregulated genes
down_plots = sorted(PATHWAY_DIR.glob('enrichment_*_down.png'))
print(f'Downregulated enrichment plots: {len(down_plots)}')

for fpath in down_plots:
    label = fpath.stem.replace('enrichment_', '').replace('_down', ' (DOWN)').replace('_', ' ')
    display(Markdown(f'**{label}**'))
    display(Image(filename=str(fpath), width=800))

## Gene Set Enrichment Analysis (GSEA)

GSEA uses IVD-specific custom gene sets to assess pathway-level changes without
requiring a significance cutoff on individual genes. The heatmap shows
normalized enrichment scores (NES) across cell types.

In [ ]:
# GSEA results summary
gsea_path = PATHWAY_DIR / 'gsea_results.tsv'
if gsea_path.exists():
    gsea = pd.read_csv(gsea_path, sep='\t')
    print(f'Total GSEA terms tested: {len(gsea)}')
    if 'padj' in gsea.columns:
        sig_gsea = gsea[gsea['padj'] < 0.05]
        print(f'Significant GSEA terms (padj < 0.05): {len(sig_gsea)}')
    elif 'pval' in gsea.columns:
        sig_gsea = gsea[gsea['pval'] < 0.05]
        print(f'Significant GSEA terms (pval < 0.05): {len(sig_gsea)}')
    
    # Show top terms
    display(gsea.head(20).style.set_caption('Top GSEA Results'))
else:
    print('GSEA results file not found')

In [ ]:
# GSEA heatmap
gsea_heatmap = PATHWAY_DIR / 'gsea_ivd_custom_heatmap.png'
if gsea_heatmap.exists():
    display(Markdown('**GSEA Heatmap — IVD Custom Gene Sets**'))
    display(Image(filename=str(gsea_heatmap), width=900))
else:
    print('GSEA heatmap not found')

## Transcription Factor Activity

TF activity is inferred from target gene expression using DoRothEA regulons.
The heatmap shows TFs with significantly altered activity between healthy and
degenerated conditions across cell types.

In [ ]:
# TF activity results
tf_path = TF_DIR / 'tf_activity_results.tsv'
if tf_path.exists():
    tf = pd.read_csv(tf_path, sep='\t')
    print(f'Total TFs tested: {tf["tf"].nunique() if "tf" in tf.columns else len(tf)}')
    
    # Count significant TFs
    for col in ['padj', 'pval', 'p_value']:
        if col in tf.columns:
            sig_tf = tf[tf[col] < 0.05]
            print(f'Significant TFs ({col} < 0.05): {len(sig_tf)}')
            display(sig_tf.head(20).style.set_caption('Top Significant TFs'))
            break
    else:
        display(tf.head(20).style.set_caption('TF Activity Results (Top 20)'))
else:
    print('TF activity results file not found')

In [ ]:
# TF activity heatmap
tf_heatmap = TF_DIR / 'tf_activity_heatmap.png'
if tf_heatmap.exists():
    display(Markdown('**TF Activity Heatmap — Healthy vs Degenerated**'))
    display(Image(filename=str(tf_heatmap), width=900))
else:
    print('TF activity heatmap not found')

## Pain-Related Gene Analysis

A curated set of pain-related genes (nociceptive signaling, inflammatory
mediators, ion channels, neurotrophins) was tested for differential expression
in degeneration. This directly addresses the clinical relevance of IVD
degeneration as a pain condition.

In [ ]:
# Pain gene results
pain_path = RESULTS / 'pain_genes.tsv'
if pain_path.exists():
    pain = pd.read_csv(pain_path, sep='\t')
    print(f'Total pain gene entries: {len(pain)}')
    print(f'Unique pain genes tested: {pain["gene"].nunique()}')
    
    # Significant pain genes
    if 'significant' in pain.columns:
        sig_pain = pain[pain['significant'] == True]
        print(f'Significant pain genes: {sig_pain["gene"].nunique()}')
        if len(sig_pain) > 0:
            cols = [c for c in ['gene', 'cell_type', 'comparison', 'log2FC', 'padj',
                                'pain_categories', 'direction'] if c in sig_pain.columns]
            display(sig_pain[cols].style.set_caption('Significant Pain Genes in Degeneration'))
    elif 'padj' in pain.columns:
        sig_pain = pain[(pain['padj'] < 0.05) & (pain['log2FC'].abs() > 0.5)]
        print(f'Significant pain genes: {sig_pain["gene"].nunique()}')
        if len(sig_pain) > 0:
            display(sig_pain.style.set_caption('Significant Pain Genes in Degeneration'))
else:
    print('Pain genes file not found')

In [ ]:
# Pain gene heatmap
pain_heatmap = RESULTS / 'pain_genes_heatmap.png'
if pain_heatmap.exists():
    display(Markdown('**Pain Gene Heatmap — Expression Across Cell Types and Conditions**'))
    display(Image(filename=str(pain_heatmap), width=900))
else:
    print('Pain genes heatmap not found')

## Status — Module 07 Complete

### Summary
- **ORA:** 1,577 significant pathway terms across all cell types
- **GSEA:** 1,576 significant terms using IVD-specific custom gene sets
- **TF activity:** 290 TFs tested, multiple with significantly altered activity
- **Pain genes:** 10 significantly DE pain genes in degeneration

### Key biological themes
- Extracellular matrix degradation pathways upregulated in degeneration
- Inflammatory and immune signaling increased in degenerated discs
- Anabolic/repair pathways (e.g., collagen synthesis) downregulated
- Pain-related genes (NGF, BDNF, inflammatory cytokines) show cell type-specific
  dysregulation in degeneration